In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"

# Data
loader = DataLoader(
    datasets.MNIST('./data', train=True, download=True,
    transform=transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])),
    batch_size=64, shuffle=True
)

# Generator
G = nn.Sequential(
    nn.Linear(100,128), nn.ReLU(),
    nn.Linear(128,256), nn.ReLU(),
    nn.Linear(256,784), nn.Tanh()
).to(device)

# Discriminator
D = nn.Sequential(
    nn.Linear(784,256), nn.LeakyReLU(0.2),
    nn.Linear(256,128), nn.LeakyReLU(0.2),
    nn.Linear(128,1), nn.Sigmoid()
).to(device)

opt_G = optim.Adam(G.parameters(), lr=0.0002)
opt_D = optim.Adam(D.parameters(), lr=0.0002)
loss_fn = nn.BCELoss()

# Train
for epoch in range(5):
    for real,_ in loader:
        real = real.view(-1,784).to(device)
        b = real.size(0)

        real_lbl = torch.ones(b,1).to(device)
        fake_lbl = torch.zeros(b,1).to(device)

        # Train D
        z = torch.randn(b,100).to(device)
        fake = G(z)
        loss_D = loss_fn(D(real), real_lbl) + loss_fn(D(fake.detach()), fake_lbl)
        opt_D.zero_grad(); loss_D.backward(); opt_D.step()

        # Train G
        z = torch.randn(b,100).to(device)
        fake = G(z)
        loss_G = loss_fn(D(fake), real_lbl)
        opt_G.zero_grad(); loss_G.backward(); opt_G.step()

    print(f"Epoch {epoch+1} | D: {loss_D.item():.4f} G: {loss_G.item():.4f}")

# Generate images
z = torch.randn(16,100).to(device)
imgs = G(z).view(-1,28,28).cpu().detach()

plt.figure(figsize=(5,5))
for i in range(16):
    plt.subplot(4,4,i+1)
    plt.imshow(imgs[i], cmap='gray')
    plt.axis('off')
plt.show()